# 经验分布函数

> 对应代码：`MathStatsCode/code_in_notes/Chap.7/empirical_distribution.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：计算对数收入的经验分布函数，并标注中位数参考线。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.7`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.7")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

载入个体数据（相对路径 `../datasets/` 指向 code_in_notes/datasets 文件夹）。


In [ ]:
%%stata
use ../datasets/chfs_ind.dta, clear


**生成对数变量**

`gen log_income=log10(labor_inc)`：生成对数收入变量。


In [ ]:
%%stata
gen log_income=log10(labor_inc)


**排序**

`sort log_income`：排序——经验分布函数要求按变量值从小到大排列后才能计算累积比例。


In [ ]:
%%stata
sort log_income


**生成排序序号，注意处理可能的缺失值**

构造经验分布的比例变量：`gen notmissing=log_income~=.` 标记非缺失观测（`~=` 是“不等于”），`qui: su notmissing` 统计有效样本数 `r(sum)`；`gen cum_log_income=_n/r(sum) if notmissing` 用“行号/总有效观测数”计算累积比例——排序后行号就是“小于等于当前值的观测个数”，因此该变量正是**经验分布函数** F̂(x) 在每个观测点的取值。


In [ ]:
%%stata
gen notmissing=log_income~=.
qui: su notmissing
gen cum_log_income=_n/r(sum) if notmissing


**画图**

`qui: su log_income, de` 取详细统计量中的中位数 `r(p50)`；用 `line` 画经验分布曲线，并用 `xline`（中位数竖虚线）、`yline(0.5, lp(dot))`（水平参考线）标注：经验分布曲线与 0.5 的水平线相交处的横坐标即中位数，直观展示两者的对应关系；`graph export` 导出图片。


In [ ]:
%%stata
qui: su log_income, de
local median_log_income=r(p50)
line cum_log_income log_income, xline(`median_log_income', lp(dot)) yline(0.5, lp(dot))
